# Step 12: Post-Training Validation and Monitoring Planning

## Purpose
This notebook defines what should happen after the churn model is deployed, including monitoring assumptions, operational checks, and retraining rules.

## What this step answers
- What kinds of data drift are most likely after deployment?
- How might model performance decay over time?
- Which metrics should be monitored in production?
- When should the model be reviewed, recalibrated, or retrained?

## Why this step matters
A packaged model is only the beginning. In production, customer behavior, pricing, plans, and acquisition patterns change. Without a monitoring plan, even a good churn model can quietly become unreliable.

In [1]:
from pathlib import Path
import json
import pandas as pd

## 1. Load the packaged model metadata

We use the metadata saved in Step 11 so the monitoring plan stays tied to the exact model that was packaged for deployment.

In [2]:
PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
metadata_file = ARTIFACTS_DIR / 'telco_churn_logistic_pipeline_metadata.json'

with open(metadata_file, 'r', encoding='utf-8') as f:
    model_metadata = json.load(f)

model_metadata

{'project': 'Telco Customer Churn',
 'step': 'Step 11 - Model Finalization and Packaging',
 'final_model_type': 'LogisticRegression',
 'comparison_model_type': 'RandomForestClassifier',
 'target_name': 'Churn',
 'target_mapping': {'No': 0, 'Yes': 1},
 'random_state': 42,
 'training_rows': 5625,
 'test_rows_held_out': 1407,
 'categorical_features': ['gender',
  'Partner',
  'Dependents',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod'],
 'numeric_features': ['SeniorCitizen',
  'tenure',
  'MonthlyCharges',
  'TotalCharges',
  'service_count',
  'avg_monthly_value_from_total',
  'is_new_customer',
  'has_long_term_contract'],
 'engineered_features': ['service_count',
  'avg_monthly_value_from_total',
  'is_new_customer',
  'has_long_term_contract'],
 'selected_hyperparameters': {'C': 1.0,
  'class_weight': 'bala

## 2. Production context assumptions

These assumptions make the monitoring plan realistic for this churn project. They are not guarantees; they are the operational risks we expect to manage.

In [3]:
assumptions_df = pd.DataFrame([
    {
        'area': 'Prediction workflow',
        'assumption': 'Predictions are generated on active customers using the same feature schema as the packaged pipeline.',
        'risk_if_broken': 'Schema mismatch or missing columns can make scores fail or become inconsistent.'
    },
    {
        'area': 'Data freshness',
        'assumption': 'Operational customer data is refreshed at least weekly or monthly, depending on business cadence.',
        'risk_if_broken': 'Delayed inputs make churn scores stale and reduce actionability.'
    },
    {
        'area': 'Target feedback',
        'assumption': 'Observed churn outcomes become available only after a delay, likely monthly or quarterly.',
        'risk_if_broken': 'Performance monitoring cannot be done in real time, so drift signals become more important.'
    },
    {
        'area': 'Business change',
        'assumption': 'Contract mix, payment behavior, pricing, and internet service mix may change over time.',
        'risk_if_broken': 'Feature distributions may shift enough that the model no longer reflects current churn drivers.'
    },
    {
        'area': 'Action threshold',
        'assumption': 'A business decision threshold will be used on top of predicted churn probability.',
        'risk_if_broken': 'Stable probabilities can still produce unstable business outcomes if the threshold is poorly managed.'
    }
] )
assumptions_df

,area,assumption,risk_if_broken
0,Prediction workflow,Predictions are generated on active customers ...,Schema mismatch or missing columns can make sc...
1,Data freshness,Operational customer data is refreshed at leas...,Delayed inputs make churn scores stale and red...
2,Target feedback,Observed churn outcomes become available only ...,Performance monitoring cannot be done in real ...
3,Business change,"Contract mix, payment behavior, pricing, and i...",Feature distributions may shift enough that th...
4,Action threshold,A business decision threshold will be used on ...,Stable probabilities can still produce unstabl...


## 3. Data drift assumptions and likely drift points

We focus on features that were important in Steps 9 and 10, because drift in high-impact features is more dangerous than drift in weak features.

In [4]:
drift_plan_df = pd.DataFrame([
    {
        'feature_or_area': 'Contract',
        'drift_expectation': 'High',
        'why_it_may_drift': 'Promotions, retention programs, and product packaging can shift the share of month-to-month versus long-term contracts.',
        'monitoring_signal': 'Population share by contract type; PSI or proportion gap versus training baseline.'
    },
    {
        'feature_or_area': 'InternetService',
        'drift_expectation': 'Medium to High',
        'why_it_may_drift': 'Fiber adoption, service availability, or product migration can change internet-service composition.',
        'monitoring_signal': 'Category distribution shift and category-specific predicted risk shift.'
    },
    {
        'feature_or_area': 'PaymentMethod',
        'drift_expectation': 'Medium',
        'why_it_may_drift': 'Billing policy changes and customer preference shifts can alter payment behavior.',
        'monitoring_signal': 'Category mix changes and calibration differences by payment method.'
    },
    {
        'feature_or_area': 'tenure / is_new_customer',
        'drift_expectation': 'High',
        'why_it_may_drift': 'Acquisition campaigns can create unusually large waves of new customers.',
        'monitoring_signal': 'Mean, median, percentile drift and share of customers with tenure <= 12.'
    },
    {
        'feature_or_area': 'MonthlyCharges / TotalCharges',
        'drift_expectation': 'Medium',
        'why_it_may_drift': 'Price updates, bundles, and taxes can change billing distributions.',
        'monitoring_signal': 'Mean/median shifts, percentile monitoring, PSI on numeric bins.'
    },
    {
        'feature_or_area': 'service_count',
        'drift_expectation': 'Medium',
        'why_it_may_drift': 'Cross-sell changes alter the number of subscribed services per customer.',
        'monitoring_signal': 'Distribution of service count and average service count over time.'
    },
    {
        'feature_or_area': 'Prediction distribution',
        'drift_expectation': 'High importance',
        'why_it_may_drift': 'Even if raw features look stable, model score distribution can move when relationships change.',
        'monitoring_signal': 'Average predicted churn probability, risk deciles, and high-risk customer share.'
    }
])
drift_plan_df

,feature_or_area,drift_expectation,why_it_may_drift,monitoring_signal
0,Contract,High,"Promotions, retention programs, and product pa...",Population share by contract type; PSI or prop...
1,InternetService,Medium to High,"Fiber adoption, service availability, or produ...",Category distribution shift and category-speci...
2,PaymentMethod,Medium,Billing policy changes and customer preference...,Category mix changes and calibration differenc...
3,tenure / is_new_customer,High,Acquisition campaigns can create unusually lar...,"Mean, median, percentile drift and share of cu..."
4,MonthlyCharges / TotalCharges,Medium,"Price updates, bundles, and taxes can change b...","Mean/median shifts, percentile monitoring, PSI..."
5,service_count,Medium,Cross-sell changes alter the number of subscri...,Distribution of service count and average serv...
6,Prediction distribution,High importance,"Even if raw features look stable, model score ...","Average predicted churn probability, risk deci..."


## 4. Expected performance decay

Since this dataset does not contain explicit time-series deployment history, we define expected decay qualitatively based on the business setting and model type.

In [5]:
performance_decay_df = pd.DataFrame([
    {
        'time_horizon': '0 to 1 month after deployment',
        'expected_behavior': 'Performance should remain close to test performance if data pipelines are stable.',
        'main_risk': 'Operational bugs, schema mismatches, or threshold misconfiguration.'
    },
    {
        'time_horizon': '1 to 3 months',
        'expected_behavior': 'Small changes in score mix and segment performance may begin to appear.',
        'main_risk': 'Acquisition mix shifts, campaign effects, and changing contract distribution.'
    },
    {
        'time_horizon': '3 to 6 months',
        'expected_behavior': 'Calibration drift and subgroup degradation become more plausible even if ranking remains useful.',
        'main_risk': 'Behavioral drift in month-to-month customers or payment-method segments.'
    },
    {
        'time_horizon': '6 months and beyond',
        'expected_behavior': 'Meaningful retraining risk; feature-target relationships may no longer match training data.',
        'main_risk': 'Business policy changes, product mix evolution, and long-term market changes.'
    }
])
performance_decay_df

,time_horizon,expected_behavior,main_risk
0,0 to 1 month after deployment,Performance should remain close to test perfor...,"Operational bugs, schema mismatches, or thresh..."
1,1 to 3 months,Small changes in score mix and segment perform...,"Acquisition mix shifts, campaign effects, and ..."
2,3 to 6 months,Calibration drift and subgroup degradation bec...,Behavioral drift in month-to-month customers o...
3,6 months and beyond,Meaningful retraining risk; feature-target rel...,"Business policy changes, product mix evolution..."


## 5. Monitoring metrics definition

A useful monitoring plan includes three layers:
- data quality checks
- drift checks
- delayed performance checks once actual churn labels arrive

In [6]:
monitoring_metrics_df = pd.DataFrame([
    {
        'metric_group': 'Data quality',
        'metric': 'Missing rate by feature',
        'why_it_matters': 'Unexpected nulls often signal broken pipelines before model performance visibly drops.',
        'frequency': 'Daily or weekly',
        'owner': 'Data / ML Ops'
    },
    {
        'metric_group': 'Data quality',
        'metric': 'Unexpected category rate',
        'why_it_matters': 'New categorical levels can appear in production and weaken model reliability.',
        'frequency': 'Weekly',
        'owner': 'Data / ML Ops'
    },
    {
        'metric_group': 'Data quality',
        'metric': 'Prediction success rate / scoring failure rate',
        'why_it_matters': 'A monitoring plan must first confirm that scoring is running consistently.',
        'frequency': 'Daily',
        'owner': 'ML Ops'
    },
    {
        'metric_group': 'Drift',
        'metric': 'Population Stability Index (or distribution gap) for key numeric features',
        'why_it_matters': 'Captures drift in tenure, charges, and service-count style features.',
        'frequency': 'Monthly',
        'owner': 'ML / Analytics'
    },
    {
        'metric_group': 'Drift',
        'metric': 'Category share drift for Contract, InternetService, PaymentMethod',
        'why_it_matters': 'These are business-sensitive and were important model drivers.',
        'frequency': 'Monthly',
        'owner': 'ML / Analytics'
    },
    {
        'metric_group': 'Drift',
        'metric': 'Average predicted churn probability',
        'why_it_matters': 'A sudden change in average model score may indicate population or concept shift.',
        'frequency': 'Weekly or monthly',
        'owner': 'ML / Business Analytics'
    },
    {
        'metric_group': 'Drift',
        'metric': 'High-risk customer share above business threshold',
        'why_it_matters': 'Directly affects intervention volume and retention operations.',
        'frequency': 'Weekly or monthly',
        'owner': 'Business + ML'
    },
    {
        'metric_group': 'Performance',
        'metric': 'Recall on newly observed labeled data',
        'why_it_matters': 'Recall was a priority metric because missing churners is costly.',
        'frequency': 'Monthly or quarterly once labels mature',
        'owner': 'ML / Analytics'
    },
    {
        'metric_group': 'Performance',
        'metric': 'Precision and intervention hit rate',
        'why_it_matters': 'Protects business teams from too many false alarms.',
        'frequency': 'Monthly or quarterly',
        'owner': 'ML / Business Analytics'
    },
    {
        'metric_group': 'Performance',
        'metric': 'ROC AUC / ranking quality',
        'why_it_matters': 'Shows whether the model still separates likely churners from non-churners well.',
        'frequency': 'Monthly or quarterly',
        'owner': 'ML / Analytics'
    },
    {
        'metric_group': 'Performance',
        'metric': 'Calibration by score band',
        'why_it_matters': 'Important if probabilities are used directly for prioritization or thresholds.',
        'frequency': 'Quarterly',
        'owner': 'ML'
    },
    {
        'metric_group': 'Fairness / segment stability',
        'metric': 'Recall and false-negative rate by key segments',
        'why_it_matters': 'Step 9 showed segment-level behavior matters, especially by contract and service groups.',
        'frequency': 'Monthly or quarterly',
        'owner': 'ML / Analytics'
    }
])
monitoring_metrics_df

,metric_group,metric,why_it_matters,frequency,owner
0,Data quality,Missing rate by feature,Unexpected nulls often signal broken pipelines...,Daily or weekly,Data / ML Ops
1,Data quality,Unexpected category rate,New categorical levels can appear in productio...,Weekly,Data / ML Ops
2,Data quality,Prediction success rate / scoring failure rate,A monitoring plan must first confirm that scor...,Daily,ML Ops
3,Drift,Population Stability Index (or distribution ga...,"Captures drift in tenure, charges, and service...",Monthly,ML / Analytics
4,Drift,"Category share drift for Contract, InternetSer...",These are business-sensitive and were importan...,Monthly,ML / Analytics
5,Drift,Average predicted churn probability,A sudden change in average model score may ind...,Weekly or monthly,ML / Business Analytics
6,Drift,High-risk customer share above business threshold,Directly affects intervention volume and reten...,Weekly or monthly,Business + ML
7,Performance,Recall on newly observed labeled data,Recall was a priority metric because missing c...,Monthly or quarterly once labels mature,ML / Analytics
8,Performance,Precision and intervention hit rate,Protects business teams from too many false al...,Monthly or quarterly,ML / Business Analytics
9,Performance,ROC AUC / ranking quality,Shows whether the model still separates likely...,Monthly or quarterly,ML / Analytics


## 6. Monitoring plan

This converts the monitoring metrics into a practical operating routine.

In [7]:
monitoring_plan_df = pd.DataFrame([
    {
        'cadence': 'Daily',
        'checks': 'Scoring job success, row counts, schema validation, scoring failure rate',
        'primary_goal': 'Catch broken pipelines before business users rely on bad scores'
    },
    {
        'cadence': 'Weekly',
        'checks': 'Missing values, unexpected categories, average predicted churn probability, high-risk customer volume',
        'primary_goal': 'Detect operational anomalies and workload spikes early'
    },
    {
        'cadence': 'Monthly',
        'checks': 'Feature drift for top predictors, segment mix drift, score distribution drift',
        'primary_goal': 'Detect population changes before they become major performance problems'
    },
    {
        'cadence': 'Quarterly',
        'checks': 'Recall, precision, F1, ROC AUC, calibration, segment-level error review',
        'primary_goal': 'Confirm the model still delivers business value once outcomes are observed'
    },
    {
        'cadence': 'Semiannual review',
        'checks': 'Full model review, threshold review, feature relevance review, retraining decision',
        'primary_goal': 'Decide whether to keep, recalibrate, or retrain the model'
    }
])
monitoring_plan_df

,cadence,checks,primary_goal
0,Daily,"Scoring job success, row counts, schema valida...",Catch broken pipelines before business users r...
1,Weekly,"Missing values, unexpected categories, average...",Detect operational anomalies and workload spik...
2,Monthly,"Feature drift for top predictors, segment mix ...",Detect population changes before they become m...
3,Quarterly,"Recall, precision, F1, ROC AUC, calibration, s...",Confirm the model still delivers business valu...
4,Semiannual review,"Full model review, threshold review, feature r...","Decide whether to keep, recalibrate, or retrai..."


## 7. Retraining triggers

Retraining should not happen only because time passed. It should happen when there is evidence that the data or performance profile has changed enough to matter.

In [8]:
retraining_triggers_df = pd.DataFrame([
    {
        'trigger_type': 'Performance drop',
        'trigger_definition': 'Recall drops by 5 percentage points or more versus test baseline for two consecutive monitoring periods.',
        'recommended_action': 'Review threshold, inspect segment failures, and start retraining workflow if confirmed.'
    },
    {
        'trigger_type': 'Ranking degradation',
        'trigger_definition': 'ROC AUC drops by 0.03 or more versus test baseline.',
        'recommended_action': 'Investigate concept drift and retrain if the drop is persistent.'
    },
    {
        'trigger_type': 'Calibration drift',
        'trigger_definition': 'Observed churn rates no longer align with predicted probability bands in a meaningful way.',
        'recommended_action': 'Recalibrate probabilities or retrain if underlying relationships changed.'
    },
    {
        'trigger_type': 'Feature drift',
        'trigger_definition': 'High-impact features such as Contract, tenure, MonthlyCharges, or InternetService show sustained material drift.',
        'recommended_action': 'Run targeted validation using recent data and retrain if score behavior also shifts.'
    },
    {
        'trigger_type': 'Segment degradation',
        'trigger_definition': 'False-negative rate becomes materially worse for important business segments such as month-to-month customers.',
        'recommended_action': 'Investigate subgroup performance and retrain if the issue persists.'
    },
    {
        'trigger_type': 'Business change',
        'trigger_definition': 'Pricing, product bundles, contract policy, or retention strategy changes substantially.',
        'recommended_action': 'Treat the model as at-risk and run a fresh validation cycle immediately.'
    },
    {
        'trigger_type': 'Time-based safeguard',
        'trigger_definition': 'No retraining for 6 to 12 months, even without visible failure signals.',
        'recommended_action': 'Run a scheduled retraining review to confirm continued relevance.'
    }
])
retraining_triggers_df

,trigger_type,trigger_definition,recommended_action
0,Performance drop,Recall drops by 5 percentage points or more ve...,"Review threshold, inspect segment failures, an..."
1,Ranking degradation,ROC AUC drops by 0.03 or more versus test base...,Investigate concept drift and retrain if the d...
2,Calibration drift,Observed churn rates no longer align with pred...,Recalibrate probabilities or retrain if underl...
3,Feature drift,"High-impact features such as Contract, tenure,...",Run targeted validation using recent data and ...
4,Segment degradation,False-negative rate becomes materially worse f...,Investigate subgroup performance and retrain i...
5,Business change,"Pricing, product bundles, contract policy, or ...",Treat the model as at-risk and run a fresh val...
6,Time-based safeguard,"No retraining for 6 to 12 months, even without...",Run a scheduled retraining review to confirm c...


## 8. Final monitoring summary for deployment handoff

This is the concise version you would hand to stakeholders or attach to deployment documentation.

In [9]:
deployment_monitoring_summary = {
    'final_model_type': model_metadata['final_model_type'],
    'primary_business_goal': 'Identify likely churners early enough for retention action',
    'primary_metric_to_watch': 'Recall on newly observed labeled churn data',
    'secondary_metrics_to_watch': ['precision', 'f1', 'roc_auc', 'calibration', 'false_negative_rate_by_segment'],
    'top_features_to_watch_for_drift': [
        'Contract',
        'tenure',
        'MonthlyCharges',
        'TotalCharges',
        'InternetService',
        'PaymentMethod',
        'service_count',
        'is_new_customer',
        'has_long_term_contract'
    ],
    'monitoring_cadence': {
        'daily': 'pipeline health and scoring success',
        'weekly': 'data quality and score volume checks',
        'monthly': 'feature drift and score drift checks',
        'quarterly': 'labeled performance review',
        'semiannual': 'formal model review and retraining decision'
    },
    'retraining_policy': 'Retrain when sustained performance degradation, material drift, or major business changes are observed, with a time-based review at least every 6 to 12 months.'
}
deployment_monitoring_summary

{'final_model_type': 'LogisticRegression',
 'primary_business_goal': 'Identify likely churners early enough for retention action',
 'primary_metric_to_watch': 'Recall on newly observed labeled churn data',
 'secondary_metrics_to_watch': ['precision',
  'f1',
  'roc_auc',
  'calibration',
  'false_negative_rate_by_segment'],
 'top_features_to_watch_for_drift': ['Contract',
  'tenure',
  'MonthlyCharges',
  'TotalCharges',
  'InternetService',
  'PaymentMethod',
  'service_count',
  'is_new_customer',
  'has_long_term_contract'],
 'monitoring_cadence': {'daily': 'pipeline health and scoring success',
  'weekly': 'data quality and score volume checks',
  'monthly': 'feature drift and score drift checks',
  'quarterly': 'labeled performance review',
  'semiannual': 'formal model review and retraining decision'},
 'retraining_policy': 'Retrain when sustained performance degradation, material drift, or major business changes are observed, with a time-based review at least every 6 to 12 month

## 9. How to use this step

Use this notebook as your deployment-readiness planning artifact:

1. `assumptions_df`
   - confirms the operating assumptions behind the packaged model

2. `drift_plan_df`
   - identifies the most likely areas of data drift

3. `monitoring_metrics_df` and `monitoring_plan_df`
   - define what should be monitored and how often

4. `retraining_triggers_df`
   - defines when the model should be reviewed or retrained

5. `deployment_monitoring_summary`
   - provides the short deployment handoff version

If you want, the next improvement after this step is to build a small production-style monitoring script that calculates these checks automatically from new scoring batches.

# Step 12 notebook setup complete.
# Run the notebook from the top and review `drift_plan_df`, `monitoring_metrics_df`, `monitoring_plan_df`, and `retraining_triggers_df`.